# Importação de Dependências

## Objetivo

Carregar as bibliotecas e módulos necessários para execução do pipeline de ingestão, transformação e persistência de dados meteorológicos utilizando Apache Spark em ambiente Databricks.

## Dependências Utilizadas

### Bibliotecas Python

| Biblioteca | Componente | Finalidade |
|------------|------------|------------|
| requests | requests | Realização de chamadas HTTP para consumo de APIs externas. |
| datetime | date, timedelta | Manipulação de datas e construção de intervalos temporais para consultas e processamento de dados. |
| uuid | uuid4 | Geração de identificadores únicos para rastreabilidade e controle de registros ou execuções. |

### Bibliotecas PySpark

| Biblioteca | Componente | Finalidade |
|------------|------------|------------|
| pyspark.sql | Row | Criação de registros estruturados em memória para posterior conversão em DataFrames. |
| pyspark.sql.functions | F | Disponibilização das funções nativas do Spark para transformação, agregação e manipulação de dados. |
| pyspark.sql.types | StructType, StructField, StringType, DoubleType, DateType | Definição explícita de schemas para garantir consistência e governança dos dados processados. |
| pyspark.sql.utils | AnalysisException | Tratamento de exceções relacionadas à análise e manipulação de objetos Spark, como tabelas e DataFrames inexistentes. |

## Considerações Técnicas

- A definição explícita das dependências melhora a legibilidade e a manutenção do código.
- O uso de schemas tipados no Spark reduz a necessidade de inferência automática de tipos, proporcionando maior previsibilidade durante a ingestão dos dados.
- A utilização de funções nativas do Spark (`pyspark.sql.functions`) é recomendada para maximizar o processamento distribuído e evitar movimentação desnecessária de dados para o driver.
- A captura de exceções específicas (`AnalysisException`) permite tratamento mais controlado de falhas operacionais e melhora a observabilidade do pipeline.

## Aplicação no Pipeline

As dependências importadas suportam as seguintes etapas do processo:

1. Consumo de dados meteorológicos via API REST.
2. Manipulação de períodos de consulta e datas de referência.
3. Criação de estruturas de dados intermediárias.
4. Transformações distribuídas utilizando Apache Spark.
5. Definição e validação de schemas.
6. Tratamento de exceções operacionais durante leitura e escrita de dados.
7. Geração de identificadores únicos para rastreabilidade e auditoria dos processos.

In [0]:
import requests
from datetime import date, timedelta
from uuid import uuid4

from pyspark.sql import Row
from pyspark.sql import functions as F  
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType, DateType)
from pyspark.sql.utils import AnalysisException

# Configurações do Pipeline de Ingestão Meteorológica

## Objetivo

Centralizar os parâmetros de configuração utilizados pelo processo de ingestão de dados meteorológicos provenientes da API Open-Meteo, garantindo padronização, reutilização e facilidade de manutenção do pipeline.

## Configurações de Armazenamento

| Variável | Valor | Descrição |
|-----------|---------|-------------|
| `TARGET_TABLE` | `gs_carbon.bronze.openmeteo_weather` | Tabela de destino responsável pelo armazenamento dos dados meteorológicos na camada Bronze. |
| `UF_COORDINATES_TABLE` | `gs_carbon.metadata.uf_coordinates` | Tabela de referência contendo as coordenadas geográficas das Unidades Federativas utilizadas nas consultas à API. |
| `TARGET_SCHEMA` | `gs_carbon.bronze` | Schema de destino onde os dados serão persistidos. |

## Configurações da Fonte de Dados

| Variável | Valor | Descrição |
|-----------|---------|-------------|
| `SOURCE_SYSTEM` | `open-meteo` | Identificação lógica do sistema de origem dos dados. |
| `SOURCE_ENDPOINT` | `https://archive-api.open-meteo.com/v1/archive` | Endpoint da API responsável pelo fornecimento dos dados meteorológicos históricos. |
| `TIMEZONE` | `America/Sao_Paulo` | Fuso horário utilizado nas consultas e padronização temporal dos dados. |
| `HISTORICAL_DAYS` | `730` | Quantidade de dias históricos considerados no processo de extração. Corresponde aproximadamente a 24 meses de dados. |

## Controle de Execução

| Variável | Descrição |
|-----------|-------------|
| `BATCH_ID` | Identificador único gerado para cada execução do pipeline, utilizado para rastreabilidade, auditoria e monitoramento dos dados processados. |

## Considerações Técnicas

- As configurações são externalizadas em variáveis para facilitar manutenção e evolução do pipeline.
- A utilização de um identificador único por execução (`BATCH_ID`) permite rastrear os registros carregados em cada processamento.
- O endpoint utilizado refere-se à API histórica da Open-Meteo, destinada à recuperação de séries temporais meteorológicas.
- A separação entre tabelas de metadados e tabelas de ingestão favorece a governança e a reutilização das informações de referência.
- O armazenamento na camada Bronze preserva os dados conforme recebidos da fonte, servindo como base para futuras transformações nas camadas Silver e Gold.

## Boas Práticas Adotadas

- Centralização de parâmetros em bloco único de configuração.
- Identificação explícita da origem dos dados.
- Separação entre configurações de infraestrutura, origem e destino.
- Uso de identificadores únicos para observabilidade e auditoria.
- Definição explícita do período histórico de coleta para garantir reprodutibilidade das cargas.

## Dependências

Este bloco depende da importação prévia da função:

```python
from uuid import uuid4
```

Responsável pela geração do identificador único utilizado na variável `BATCH_ID`.

In [0]:
TARGET_TABLE = "gs_carbon.bronze.openmeteo_weather"
UF_COORDINATES_TABLE = "gs_carbon.metadata.uf_coordinates"
TARGET_SCHEMA = "gs_carbon.bronze"

SOURCE_SYSTEM = "open-meteo"
SOURCE_ENDPOINT = "https://archive-api.open-meteo.com/v1/archive"
TIMEZONE = "America/Sao_Paulo"
HISTORICAL_DAYS = 730

BATCH_ID = str(uuid4())

# Definição do Schema de Dados Meteorológicos

## Objetivo

Definir a estrutura padrão dos dados meteorológicos ingeridos a partir da API Open-Meteo, garantindo consistência de tipagem, validação de campos obrigatórios e conformidade dos registros processados ao longo do pipeline.

## Descrição

O schema é utilizado para estruturar os dados carregados no Apache Spark, estabelecendo os tipos de dados esperados para cada atributo meteorológico e geográfico.

A adoção de um schema explícito reduz a dependência da inferência automática de tipos, melhora a qualidade dos dados e aumenta a previsibilidade dos processos de transformação e persistência.

## Estrutura dos Dados

| Campo | Tipo | Obrigatório | Descrição |
|---------|---------|---------|---------|
| `uf` | String | Sim | Unidade Federativa associada à observação meteorológica. |
| `latitude` | Double | Não | Latitude da localização consultada. |
| `longitude` | Double | Não | Longitude da localização consultada. |
| `reference_date` | String | Sim | Data de referência da observação meteorológica. |
| `temperature_mean` | Double | Não | Temperatura média diária registrada (°C). |
| `precipitation_sum` | Double | Não | Volume total de precipitação acumulada no período (mm). |
| `wind_speed_max` | Double | Não | Velocidade máxima do vento registrada no período (km/h). |

## Regras de Validação

### Campos Obrigatórios

Os seguintes atributos não permitem valores nulos:

- `uf`
- `reference_date`

A ausência dessas informações invalida o registro para fins de processamento.

### Campos Opcionais

Os atributos meteorológicos e geográficos podem ser nulos em situações onde:

- A fonte de dados não disponibilize a informação para determinada data ou localidade.
- Haja indisponibilidade parcial dos dados retornados pela API.
- O processo de coleta identifique registros incompletos provenientes da origem.

## Considerações Técnicas

- O schema utiliza `StructType` e `StructField` para garantir controle explícito sobre a estrutura dos dados.
- A tipagem numérica é definida como `DoubleType`, permitindo armazenar valores decimais com precisão adequada para indicadores meteorológicos.
- A definição antecipada do schema melhora a performance de leitura ao evitar processos de inferência automática executados pelo Spark.
- A utilização de schemas explícitos contribui para a governança e rastreabilidade dos ativos de dados.

## Recomendações

- Avaliar a utilização de `DateType` para o campo `reference_date` caso sejam realizadas operações analíticas ou transformações temporais diretamente no Spark.
- Implementar validações adicionais de qualidade para monitoramento de valores nulos em métricas meteorológicas críticas.
- Versionar alterações estruturais do schema para garantir compatibilidade entre diferentes versões do pipeline.

## Aplicação no Pipeline

Este schema é utilizado durante a etapa de ingestão para:

1. Estruturar os dados retornados pela API Open-Meteo.
2. Validar a conformidade dos registros recebidos.
3. Criar DataFrames Spark com tipagem controlada.
4. Garantir consistência dos dados persistidos na camada Bronze.

In [0]:
weather_schema = StructType(
    [
        StructField("uf", StringType(), False),
        StructField("latitude", DoubleType(), True),
        StructField("longitude", DoubleType(), True),
        StructField("reference_date", StringType(), False),
        StructField("temperature_mean", DoubleType(), True),
        StructField("precipitation_sum", DoubleType(), True),
        StructField("wind_speed_max", DoubleType(), True),
    ]
)

# Controle de Carga e Consumo da API Open-Meteo

## Objetivo

Estas funções são responsáveis por controlar o período de extração dos dados meteorológicos e realizar o consumo da API Open-Meteo.

A função `get_start_date()` implementa a estratégia de carga incremental do pipeline, identificando a última data processada na tabela de destino. Caso a tabela não exista ou esteja vazia, o processo executa automaticamente uma carga histórica considerando a janela definida em `HISTORICAL_DAYS`.

A função `get_weather_history()` realiza a consulta à API Open-Meteo para obtenção dos indicadores meteorológicos históricos, utilizando latitude, longitude e intervalo de datas como parâmetros de entrada.

## Principais Regras

- Cargas incrementais iniciam a partir do dia seguinte ao último registro processado.
- Na ausência de dados previamente carregados, é executada uma carga histórica.
- As consultas à API retornam temperatura média, precipitação acumulada e velocidade máxima do vento.
- Respostas inválidas da API geram exceção por meio da validação HTTP (`raise_for_status`).

## Resultado Esperado

Ao final da execução, o pipeline obtém os dados meteorológicos necessários para o período pendente de processamento, evitando reprocessamento de informações já persistidas e garantindo eficiência na ingestão incremental.

In [0]:
def get_start_date(target_table: str) -> str:
    try:
        max_date = (
            spark.table(target_table)
            .agg(F.max("reference_date").alias("max_date"))
            .collect()[0]["max_date"]
        )

        if max_date is None:
            start_date = date.today() - timedelta(days=HISTORICAL_DAYS)
            print(f"Tabela vazia. Executando carga historica desde {start_date}")
            return start_date.strftime("%Y-%m-%d")

        start_date = max_date + timedelta(days=1)
        print(f"Carga incremental iniciada em {start_date}")
        return start_date.strftime("%Y-%m-%d")

    except AnalysisException:
        start_date = date.today() - timedelta(days=HISTORICAL_DAYS)
        print(f"Tabela inexistente. Executando carga historica desde {start_date}")
        return start_date.strftime("%Y-%m-%d")


def get_weather_history(latitude, longitude, start_date, end_date):
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "daily": "temperature_2m_mean,precipitation_sum,wind_speed_10m_max",
        "timezone": TIMEZONE,
    }

    response = requests.get(SOURCE_ENDPOINT, params=params, timeout=120)
    response.raise_for_status()

    return response.json()

# Conversão Segura de Valores Numéricos

## Objetivo

Função utilitária responsável por padronizar a conversão de valores para o tipo `float`, preservando valores nulos quando presentes na origem dos dados.

## Regra Aplicada

- Valores diferentes de `None` são convertidos para `float`.
- Valores nulos (`None`) são mantidos sem alteração.

## Aplicação no Pipeline

Esta função auxilia o tratamento dos dados retornados pela API, garantindo consistência na tipagem dos campos numéricos e evitando erros durante a criação dos DataFrames Spark.

In [0]:
def to_float_or_none(value):
    return float(value) if value is not None else None

# Extração e Estruturação dos Dados Meteorológicos

## Objetivo

Responsável por consumir os dados meteorológicos para cada Unidade Federativa (UF) e transformar o retorno da API em registros estruturados para processamento no Spark.

## Principais Regras

- Realiza consultas individuais para cada UF utilizando suas coordenadas geográficas.
- Converte os dados diários retornados pela API em objetos `Row`, seguindo o schema definido para o pipeline.
- Aplica tratamento de tipagem nos indicadores meteorológicos para garantir consistência dos dados.
- Registra falhas de extração sem interromper o processamento das demais UFs.

## Resultado Esperado

A função retorna uma coleção de registros meteorológicos prontos para criação do DataFrame Spark, além de uma lista contendo eventuais UFs que apresentaram erro durante a extração para fins de monitoramento e reprocessamento.# Extração e Estruturação dos Dados Meteorológicos

## Objetivo

Responsável por consumir os dados meteorológicos para cada Unidade Federativa (UF) e transformar o retorno da API em registros estruturados para processamento no Spark.

## Principais Regras

- Realiza consultas individuais para cada UF utilizando suas coordenadas geográficas.
- Converte os dados diários retornados pela API em objetos `Row`, seguindo o schema definido para o pipeline.
- Aplica tratamento de tipagem nos indicadores meteorológicos para garantir consistência dos dados.
- Registra falhas de extração sem interromper o processamento das demais UFs.

## Resultado Esperado

A função retorna uma coleção de registros meteorológicos prontos para criação do DataFrame Spark, além de uma lista contendo eventuais UFs que apresentaram erro durante a extração para fins de monitoramento e reprocessamento.

In [0]:
def extract_weather_rows(uf_rows, start_date, end_date):
    rows = []
    failed_ufs = []

    for uf in uf_rows:
        print(f"Processando {uf.uf}")

        try:
            payload = get_weather_history(
                latitude=uf.latitude,
                longitude=uf.longitude,
                start_date=start_date,
                end_date=end_date,
            )

            daily = payload.get("daily", {})
            dates = daily.get("time", [])

            for idx, reference_date in enumerate(dates):
                rows.append(
                    Row(
                        uf=uf.uf,
                        latitude=float(uf.latitude),
                        longitude=float(uf.longitude),
                        reference_date=reference_date,
                        temperature_mean=to_float_or_none(
                            daily.get("temperature_2m_mean", [None])[idx]
                        ),
                        precipitation_sum=to_float_or_none(
                            daily.get("precipitation_sum", [None])[idx]
                        ),
                        wind_speed_max=to_float_or_none(
                            daily.get("wind_speed_10m_max", [None])[idx]
                        ),
                    )
                )

        except Exception as exc:
            failed_ufs.append((uf.uf, str(exc)))
            print(f"Erro na UF {uf.uf}: {str(exc)}")

    return rows, failed_ufs

# Processamento e Persistência dos Dados Meteorológicos

## Objetivo

Executar o fluxo de carga dos dados meteorológicos, desde a definição do período de extração até a persistência na camada Bronze utilizando tabelas Delta.

## Principais Regras

- O processamento ocorre apenas quando existem datas pendentes de carga.
- Os dados extraídos são enriquecidos com metadados de ingestão, rastreabilidade e origem.
- Registros duplicados são eliminados considerando a combinação de `uf` e `reference_date`.
- A persistência utiliza operação `MERGE`, garantindo comportamento de upsert para atualização de registros existentes e inserção de novos dados.
- A tabela de destino é criada automaticamente caso ainda não exista.

## Resultado Esperado

Ao final da execução, a tabela `openmeteo_weather` é atualizada com os dados mais recentes disponíveis na API, preservando a integridade dos registros e a rastreabilidade da carga por meio de informações como lote, período processado e origem dos dados.

In [0]:
START_DATE = get_start_date(TARGET_TABLE)
END_DATE = date.today().strftime("%Y-%m-%d")

print(f"Fim da carga: {END_DATE}")

if START_DATE > END_DATE:
    print("Nenhuma nova data para carregar.")

else:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}")

    uf_rows = (
        spark.table(UF_COORDINATES_TABLE)
        .select("uf", "latitude", "longitude")
        .dropDuplicates(["uf"])
        .collect()
    )

    rows, failed_ufs = extract_weather_rows(
        uf_rows=uf_rows,
        start_date=START_DATE,
        end_date=END_DATE,
    )

    if not rows:
        print("Nenhum registro retornado pela API.")

    else:
        weather_df = (
            spark.createDataFrame(rows, schema=weather_schema)
            .withColumn("reference_date", F.to_date("reference_date"))
            .withColumn("ingestion_timestamp", F.current_timestamp())
            .withColumn("ingestion_date", F.current_date())
            .withColumn("batch_id", F.lit(BATCH_ID))
            .withColumn("source_system", F.lit(SOURCE_SYSTEM))
            .withColumn("source_endpoint", F.lit(SOURCE_ENDPOINT))
            .withColumn("load_start_date", F.lit(START_DATE).cast("date"))
            .withColumn("load_end_date", F.lit(END_DATE).cast("date"))
            .dropDuplicates(["uf", "reference_date"])
        )

        weather_df.createOrReplaceTempView("vw_openmeteo_weather_staging")

        spark.sql(
            f"""
            CREATE TABLE IF NOT EXISTS {TARGET_TABLE}
            USING DELTA
            AS
            SELECT *
            FROM vw_openmeteo_weather_staging
            WHERE 1 = 0
            """
        )

        spark.sql(
            f"""
            MERGE INTO {TARGET_TABLE} AS target
            USING vw_openmeteo_weather_staging AS source
            ON target.uf = source.uf
               AND target.reference_date = source.reference_date
            WHEN MATCHED THEN UPDATE SET
                target.latitude = source.latitude,
                target.longitude = source.longitude,
                target.temperature_mean = source.temperature_mean,
                target.precipitation_sum = source.precipitation_sum,
                target.wind_speed_max = source.wind_speed_max,
                target.ingestion_timestamp = source.ingestion_timestamp,
                target.ingestion_date = source.ingestion_date,
                target.batch_id = source.batch_id,
                target.source_system = source.source_system,
                target.source_endpoint = source.source_endpoint,
                target.load_start_date = source.load_start_date,
                target.load_end_date = source.load_end_date
            WHEN NOT MATCHED THEN INSERT *
            """
        )

        loaded_records = weather_df.count()

        print(f"Registros processados: {loaded_records}")
        print(f"Tabela atualizada: {TARGET_TABLE}")

    if failed_ufs:
        print(f"UFs com erro: {failed_ufs}")

In [0]:
%sql
DESCRIBE HISTORY gs_carbon.bronze.openmeteo_weather

In [0]:
%sql
SELECT * FROM gs_carbon.bronze.openmeteo_weather